In [ ]:
#imports
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import random
from pathlib import Path
import os 
import sys
from collections import Counter


In [ ]:
#reload notebooks
%load_ext autoreload
%autoreload 2

In [ ]:
#Set up parent directory variable
BASE_DIR = Path.cwd() 
print(BASE_DIR)

c:\Users\Neezy\Videos\attacks_improved_v2\Attacks-Improved_v2\decision_tree


In [4]:
project_root = os.path.abspath('..')
if project_root not in sys.path: 
    sys.path.append(project_root)

In [ ]:
#Read dataset 
data = pd.read_csv(BASE_DIR.parent/'data/ds_with_sample_name.csv')
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [6]:
# import attacks
from attack_functionality.functionality_preserving_attack import FunctionalityPreservingAttack 
attack = FunctionalityPreservingAttack()

In [ ]:
#custom features
features = ['LdrGetProcedureAddress', 'GetSystemInfo', 'GetNativeSystemInfo', 'NtQuerySystemInformation',  'CreateToolhelp32Snapshot', 'Process32FirstW', 'RegOpenKeyExW', 'RegQueryValueExW',
            'RegCloseKey', 'NtCreateFile', 'GetDiskFreeSpaceW', 'CryptAcquireContextW', 'CryptGenKey', 
            'CryptExportKey', 'RegCreateKeyExW', 'RegSetValueExW', 'CreateThread', 'FindFirstFileExW',
            'NtWriteFile', 'ShellExecuteExW', 'InternetOpenW', 
            'InternetConnectW', 'HttpOpenRequestW', 'InternetReadFile', 'InternetOpenUrlA', 
            'InternetCloseHandle']

In [ ]:
#custom features
benign_features = ['LdrGetDllHandle',
 'RegOpenKeyExW',
 'RegQueryValueExW',
 'RegCloseKey',
 'GetSystemWindowsDirectoryW',
 'CreateActCtxW',
 'GetSystemDirectoryW',
 'GetVolumeNameForVolumeMountPointW',
 'RegEnumKeyW',
 'GlobalMemoryStatusEx',
 'CreateDirectoryW',
 'GetVolumePathNamesForVolumeNameW',
 'LdrUnloadDll',
 'GetFileAttributesW',
 'SHGetFolderPathW',
 'GetSystemMetrics',
 'SetFileTime',
 'RegEnumKeyExW',
 'DrawTextExW',
 'FindWindowW',
 'FindWindowExA',
 'RemoveDirectoryW',
 'GetKeyboardState',
 'SendNotifyMessageW',
 'GetFileVersionInfoSizeW',
 'EnumWindows',
 'GetUserNameExA',
 'RegCreateKeyExA',
 'SetInformationJobObject',
 'DrawTextExA',
 'RegSetValueExA',
 'SHGetSpecialFolderLocation',
 'gethostbyname',
 'CertOpenStore',
 'CertControlStore',
 'SetWindowsHookExW',
 'WSASocketW',
 'WSAConnect',
 'UnhookWindowsHookEx',
 'CertOpenSystemStoreW',
 'HttpQueryInfoA',
 'CryptProtectMemory',
 'WriteProcessMemory',
 'SendNotifyMessageA',
 'JsGlobalObjectDefaultEvalHelper',
 'ObtainUserAgentString',
 'label'
 ]

In [ ]:
# no duplicate features
benign_filtered = [item for item in benign_features if item not in features]

In [10]:
len(benign_filtered)

44

In [ ]:
#full custom featureset 
functionality_features = features.extend(benign_filtered)

In [12]:
print(len(features))

70


In [13]:
for i,v in enumerate(features): 
    if i>=26: 
        print(i,v)

26 LdrGetDllHandle
27 GetSystemWindowsDirectoryW
28 CreateActCtxW
29 GetSystemDirectoryW
30 GetVolumeNameForVolumeMountPointW
31 RegEnumKeyW
32 GlobalMemoryStatusEx
33 CreateDirectoryW
34 GetVolumePathNamesForVolumeNameW
35 LdrUnloadDll
36 GetFileAttributesW
37 SHGetFolderPathW
38 GetSystemMetrics
39 SetFileTime
40 RegEnumKeyExW
41 DrawTextExW
42 FindWindowW
43 FindWindowExA
44 RemoveDirectoryW
45 GetKeyboardState
46 SendNotifyMessageW
47 GetFileVersionInfoSizeW
48 EnumWindows
49 GetUserNameExA
50 RegCreateKeyExA
51 SetInformationJobObject
52 DrawTextExA
53 RegSetValueExA
54 SHGetSpecialFolderLocation
55 gethostbyname
56 CertOpenStore
57 CertControlStore
58 SetWindowsHookExW
59 WSASocketW
60 WSAConnect
61 UnhookWindowsHookEx
62 CertOpenSystemStoreW
63 HttpQueryInfoA
64 CryptProtectMemory
65 WriteProcessMemory
66 SendNotifyMessageA
67 JsGlobalObjectDefaultEvalHelper
68 ObtainUserAgentString
69 label


In [14]:
df = data[features]

In [15]:
df.head()

,LdrGetProcedureAddress,GetSystemInfo,GetNativeSystemInfo,NtQuerySystemInformation,CreateToolhelp32Snapshot,Process32FirstW,RegOpenKeyExW,RegQueryValueExW,RegCloseKey,NtCreateFile,GetDiskFreeSpaceW,CryptAcquireContextW,CryptGenKey,CryptExportKey,RegCreateKeyExW,RegSetValueExW,CreateThread,FindFirstFileExW,NtWriteFile,ShellExecuteExW,InternetOpenW,InternetConnectW,HttpOpenRequestW,InternetReadFile,InternetOpenUrlA,InternetCloseHandle,LdrGetDllHandle,GetSystemWindowsDirectoryW,CreateActCtxW,GetSystemDirectoryW,GetVolumeNameForVolumeMountPointW,RegEnumKeyW,GlobalMemoryStatusEx,CreateDirectoryW,GetVolumePathNamesForVolumeNameW,LdrUnloadDll,GetFileAttributesW,SHGetFolderPathW,GetSystemMetrics,SetFileTime,RegEnumKeyExW,DrawTextExW,FindWindowW,FindWindowExA,RemoveDirectoryW,GetKeyboardState,SendNotifyMessageW,GetFileVersionInfoSizeW,EnumWindows,GetUserNameExA,RegCreateKeyExA,SetInformationJobObject,DrawTextExA,RegSetValueExA,SHGetSpecialFolderLocation,gethostbyname,CertOpenStore,CertControlStore,SetWindowsHookExW,WSASocketW,WSAConnect,UnhookWindowsHookEx,CertOpenSystemStoreW,HttpQueryInfoA,CryptProtectMemory,WriteProcessMemory,SendNotifyMessageA,JsGlobalObjectDefaultEvalHelper,ObtainUserAgentString,label
0,1426,10,9,7,0,0,1547,1308,7834,7398,0,1,0,0,235,271,6,17478,40704,5,0,0,0,2,2,7,141,58,6,25,21,8,1,9,48,74,149,8,670,3317,128,86,0,0,0,0,0,9,6,0,4,0,0,4,6,0,0,0,0,0,0,0,0,0,4,3,0,0,0,1
1,24,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,11,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
2,564,162,1,4,0,0,195,317,136,1213,0,213,0,0,2,1,7,0,351,1,0,0,0,0,0,0,200,10,5,11,3,0,13,3,8,7,25,11,6,0,38,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1
3,245,92,0,2,0,0,106,182,66,95,0,0,0,0,1,0,4,0,2244,0,0,0,0,0,0,0,70,2,3,4,0,0,13,4,0,2,11,2,1,0,21,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,3,1,0,0,0,0,0,0,0,0,1
4,5,2,0,0,0,0,0,0,1,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1


In [ ]:
#Train surrogate model 
X = df.iloc[:,:-1].values
y = df.iloc[:,-1].values
original_indices = np.arange(len(X)) 
X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(X, y, original_indices, test_size=0.2, random_state=42)
classifier = DecisionTreeClassifier(random_state=42)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy * 100:.2f}%')
cm = confusion_matrix(y_test, y_pred)
print(f"Confusion Matrix:\n {cm}") 
cr = classification_report(y_test, y_pred)
print(f"Classification report:\n {cr}")

Accuracy: 96.91%
Confusion Matrix:
 [[105  12]
 [  6 460]]
Classification report:
               precision    recall  f1-score   support

           0       0.95      0.90      0.92       117
           1       0.97      0.99      0.98       466

    accuracy                           0.97       583
   macro avg       0.96      0.94      0.95       583
weighted avg       0.97      0.97      0.97       583



In [17]:
def make_pred(sample): 
    single_pred = classifier.predict([sample])
    return single_pred 

In [ ]:
# ransomware_training_sample_indices = np.where(y_train==1)[0]
# non_zero_mask = X_train[ransomware_training_sample_indices].sum(axis=1) !=0
# valid_indices = ransomware_training_sample_indices[non_zero_mask]

# idx = np.random.choice(valid_indices, 500, replace=False)
# xtrain_poison_candidates = X_train[idx] 
idx_loaded = np.load(BASE_DIR.parent/"Adversarial_candidates_indices/dt_adversarial_candidates_indices.npy")
xtrain_poison_candidates = X_train[idx_loaded] 
print(len(xtrain_poison_candidates))

580


In [19]:
print(len(xtrain_poison_candidates))

580


In [ ]:
#define perturbation value
perturbation=10
#Generate adversarial samples
asamples, adv_idx = attack.generate_adversarial_sample(features, idx_loaded, xtrain_poison_candidates, perturbation, classifier)

["Feature: CreateDirectoryW changed: {'from': 0, 'to': 2}"]
["Feature: RegQueryValueExW changed: {'from': 0, 'to': 4}", "Feature: CreateDirectoryW changed: {'from': 0, 'to': 2}", "Feature: LdrUnloadDll changed: {'from': 0, 'to': 2}"]
["Feature: CreateDirectoryW changed: {'from': 0, 'to': 2}"]
["Feature: NtCreateFile changed: {'from': 0, 'to': 1}", "Feature: CreateDirectoryW changed: {'from': 0, 'to': 2}"]
["Feature: RegQueryValueExW changed: {'from': 0, 'to': 4}", "Feature: CreateDirectoryW changed: {'from': 0, 'to': 2}", "Feature: LdrUnloadDll changed: {'from': 0, 'to': 2}"]
["Feature: RegQueryValueExW changed: {'from': 0, 'to': 4}", "Feature: CreateDirectoryW changed: {'from': 0, 'to': 2}", "Feature: LdrUnloadDll changed: {'from': 0, 'to': 2}"]
["Feature: CreateDirectoryW changed: {'from': 0, 'to': 2}"]
["Feature: CreateDirectoryW changed: {'from': 0, 'to': 2}"]
["Feature: CreateDirectoryW changed: {'from': 0, 'to': 2}"]
["Feature: RegQueryValueExW changed: {'from': 0, 'to': 4}", "Fe

In [21]:
print(len(asamples))
print(f"AGR: {len(asamples)/len(xtrain_poison_candidates):.4f}") 

370
AGR: 0.6379


In [23]:
### Adversarial Retrain ###  
mask = np.ones(X_train.shape[0], dtype=bool)
mask[adv_idx] = False

X_train_base_samples_removed = X_train[mask]
y_train_base_samples_removed = y_train[mask]
print(f"length of Xtrain before successful base adversarial samples removed: {len(X_train)}")
print(f"length of Xtrain with successful base adversarial samples removed: {len(X_train_base_samples_removed)}")

### Select a hold out test set of the generated adversarial samples
k = round(0.2 * len(asamples)) 
print(f"20% of the adversarial samples held out for testing: {k}")
remove_samples_from_asamples = random.sample(asamples, k)
print(len(remove_samples_from_asamples))

### remove test set from total adversarial samples and add adversarial samples to the training set 
result = [item for item in asamples if item not in remove_samples_from_asamples]
np_result = np.array(result)
print(len(result))
poisoned_xtrain = np.concatenate((X_train_base_samples_removed, np_result))
print(f"Total length of poisoned Xtrain: {len(poisoned_xtrain)}")

### Add the corresponding amount of ytrain adversarial samples. (correctly labelled as ransomware) 
adv_y = []
for i in range(len(np_result)): 
    adv_y.append(1) 
    
print(len(adv_y))

np_advy = np.array(adv_y)
poisoned_ytrain = np.concatenate((y_train_base_samples_removed, np_advy))
print(f"Total length of poisoned Ytrain: {len(poisoned_ytrain)}") 

if len(poisoned_ytrain) == len(poisoned_xtrain): 
    print("Poisoned Xtrain and ytrain are aligned, posioned training can now commence")
else: 
    print("Not aligned, something is not aligned, check k value")

length of Xtrain before successful base adversarial samples removed: 2330
length of Xtrain with successful base adversarial samples removed: 1960
20% of the adversarial samples held out for testing: 74
74
72
Total length of poisoned Xtrain: 2032
72
Total length of poisoned Ytrain: 2032
Poisoned Xtrain and ytrain are aligned, posioned training can now commence


In [ ]:
#Retrain surrogate model with adversarial samples added to the training set 
poisoned_classifier = DecisionTreeClassifier(random_state=42)
poisoned_classifier.fit(poisoned_xtrain, poisoned_ytrain)
y_pred_poisoned = poisoned_classifier.predict(X_test)
poisoned_accuracy = accuracy_score(y_test, y_pred_poisoned)
print(f'Accuracy trained on poisoned samples: {poisoned_accuracy * 100:.2f}%')
cm_poisoned = confusion_matrix(y_test, y_pred_poisoned)
print(f"Confusion Matrix trained on poisoned samples:\n {cm_poisoned}") 
cr_poisoned = classification_report(y_test, y_pred_poisoned)
print(f"Classification report trained on poisoned samples:\n {cr_poisoned}")

Accuracy trained on poisoned samples: 96.91%
Confusion Matrix trained on poisoned samples:
 [[104  13]
 [  5 461]]
Classification report trained on poisoned samples:
               precision    recall  f1-score   support

           0       0.95      0.89      0.92       117
           1       0.97      0.99      0.98       466

    accuracy                           0.97       583
   macro avg       0.96      0.94      0.95       583
weighted avg       0.97      0.97      0.97       583



In [ ]:
#Evaluate retrained surrogate model
def make_pred_poisoned(samples):
    predictions = [] 
    for sample in samples:
        single_pred = poisoned_classifier.predict([sample])
        predictions.append(single_pred)
        print(single_pred) 
        if single_pred[0] == 1: 
            print("Correctly identified adversarial ransomware sample") 
        else: 
            print("Adversarial sample still adversarial")

    zero_counter = 0 
    one_counter = 0 
    for pred in predictions:
        if pred != 1: 
            zero_counter +=1 
        else:
            one_counter+=1 
        
    
    print(f"Samples still adversarial: {zero_counter}")
    print(f"adversarial_samples identified: {one_counter}")

make_pred_poisoned(remove_samples_from_asamples)

[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correc

In [26]:
###Model transfer ###
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

In [27]:
### Target Models ####
##  Trained on original data
#   No adversarial samples are added to the training data 

### Random Forest ####
classifier_rf = RandomForestClassifier(n_estimators=100, random_state=42)
classifier_rf.fit(X_train, y_train)
y_pred_rf = classifier_rf.predict(X_test)
accuracy_rf = accuracy_score(y_test, y_pred_rf) 
print(f"Accuracy of Random Forest Target Model: {accuracy_score(y_test, y_pred):.2f}")

### XGboost #### 
params = {
    'objective':'binary:logistic',
    'max_depth':4,
    'learning_rate':0.1,
    'n_estimators':100,
    'alpha':10
}

model = XGBClassifier(**params)
model.fit(X_train, y_train)
y_pred_xgb = model.predict(X_test)
accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
print(f'Accuracy of the XGboost Model: {accuracy_xgb * 100:.2f}%') 

### SVM Classifier Kernel=Poly ###
svm_classifier = SVC(kernel='poly', C=1.0, random_state=42, max_iter=5000)
svm_classifier.fit(X_train, y_train)
y_pred_SVM_poly = svm_classifier.predict(X_test)
print(f"Accuracy of the SVM Classifier (Kernel=Poly): {accuracy_score(y_test, y_pred_SVM_poly):.2f}")

### SVM Classifier Kernel=Linear ###
svm_classifier_linear = SVC(kernel='linear', C=4.0, random_state=42, max_iter=5000)
svm_classifier_linear.fit(X_train, y_train)
y_pred_SVM_linear = svm_classifier_linear.predict(X_test)
print(f"Accuracy of the SVM Classifier (Kernel=Linear) : {accuracy_score(y_test, y_pred_SVM_linear):.2f}")

### Logistic Regression ###
LR_model = LogisticRegression(solver='saga',max_iter=5000)
LR_model.fit(X_train, y_train)
ypred_lr = LR_model.predict(X_test)
accuracy_lr = accuracy_score(y_test, ypred_lr)
print(f"accuracy of the Logistic Regression Model: {round(accuracy_lr,2)}")

Accuracy of Random Forest Target Model: 0.97
Accuracy of the XGboost Model: 97.60%
Accuracy of the SVM Classifier (Kernel=Poly): 0.83


c:\Users\Neezy\anaconda3\envs\tf\lib\site-packages\sklearn\svm\_base.py:305: ConvergenceWarning: Solver terminated early (max_iter=5000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(


Accuracy of the SVM Classifier (Kernel=Linear) : 0.95
accuracy of the Logistic Regression Model: 0.95


In [28]:
def evasion_attack(adversarial_samples,
                   RF_model,
                   XGB_Model, 
                   SVM_poly_model, 
                   SVM_linear,
                   Logistic_reg_model):
    
    predictions = {
        "Random_forest":[],
        "XGB":[],
        "SVM_poly": [], 
        "SVM_Linear":[], 
        "Logistic_Regression":[]   
    }
    
    models = {
        "Random_forest": RF_model,
        "XGB":XGB_Model,
        "SVM_poly": SVM_poly_model,
        "SVM_Linear":SVM_linear, 
        "Logistic_Regression":Logistic_reg_model
    }
    
    for name, model in models.items():
        preds = model.predict(adversarial_samples)
        predictions[name] = preds.tolist()
            
    
    return predictions

In [29]:
res = evasion_attack(asamples, classifier_rf, model, svm_classifier, svm_classifier_linear, LR_model)
evaded_ids=[]
evaded_samples = []
original_positions = []
for model_name, preds in res.items():
    print(f"\n {model_name}: predictions")
    total = len(preds)
    count_1 = sum(1 for p in preds if p==1)
    count_2 = total-count_1 
    
    model_evaded_positions = []
    model_evaded_samples = []
    
    for i, p in enumerate(preds): 
        if p == 0: 
           xtrain_position = adv_idx[i] 
           original_position = idx_train[xtrain_position]
           model_evaded_positions.append(original_position)
            
    if model_evaded_positions:
        for og_position in model_evaded_positions:
            og_sample = data.iloc[og_position]['Sample Name']
            model_evaded_samples.append(og_sample)
        evaded_samples.extend(model_evaded_samples)
        original_positions.extend(model_evaded_positions)
    else: 
        print(f"\n No samples evaded for {model_name}")
    
    print(f"\n {model_name} evasion rate: {count_2}/{total} samples evaded")
    
    sample_names = [s.rsplit('_', 1)[0] for s in model_evaded_samples]
    print("\n evaded samples: ", Counter(sample_names))      


 Random_forest: predictions

 No samples evaded for Random_forest

 Random_forest evasion rate: 0/370 samples evaded

 evaded samples:  Counter()

 XGB: predictions

 No samples evaded for XGB

 XGB evasion rate: 0/370 samples evaded

 evaded samples:  Counter()

 SVM_poly: predictions

 No samples evaded for SVM_poly

 SVM_poly evasion rate: 0/370 samples evaded

 evaded samples:  Counter()

 SVM_Linear: predictions

 No samples evaded for SVM_Linear

 SVM_Linear evasion rate: 0/370 samples evaded

 evaded samples:  Counter()

 Logistic_Regression: predictions

 Logistic_Regression evasion rate: 10/370 samples evaded

 evaded samples:  Counter({'None': 3, 'Downloader-CUZ': 2, 'BlackCat': 2, 'Magniber': 1, 'WannaCry': 1, 'NotPetya': 1})


In [30]:
scaler = StandardScaler()
Xtrain_scaled = scaler.fit_transform(X_train)
Xtest_scaled = scaler.transform(X_test)

In [31]:
### Scaled Target Models ####
##  Trained on original data
#   No adversarial samples are added to the training data 

### Random Forest ####
classifier_rf_scaled = RandomForestClassifier(n_estimators=100, random_state=42)
classifier_rf_scaled.fit(Xtrain_scaled, y_train)
y_pred_rf_scaled = classifier_rf_scaled.predict(Xtest_scaled)
accuracy_rf_scaled = accuracy_score(y_test, y_pred_rf_scaled) 
print(f"Accuracy of Random Forest Target Model: {accuracy_rf_scaled:.2f}")

### XGboost #### 
params = {
    'objective':'binary:logistic',
    'max_depth':4,
    'learning_rate':0.1,
    'n_estimators':100,
    'alpha':10
}

model_scaled = XGBClassifier(**params)
model_scaled.fit(Xtrain_scaled, y_train)
y_pred_xgb_scaled = model_scaled.predict(Xtest_scaled)
accuracy_xgb_scaled = accuracy_score(y_test, y_pred_xgb_scaled)
print(f'Accuracy of the XGboost Model: {accuracy_xgb_scaled * 100:.2f}%') 

### SVM Classifier Kernel=Poly ###
svm_classifier_scaled = SVC(kernel='poly', C=1.0, random_state=42)
svm_classifier_scaled.fit(Xtrain_scaled, y_train)
y_pred_SVM_poly_scaled = svm_classifier_scaled.predict(Xtest_scaled)
print(f"Accuracy of the SVM Classifier (Kernel=Poly): {accuracy_score(y_test, y_pred_SVM_poly_scaled):.2f}")

### SVM Classifier Kernel=Linear ###
svm_classifier_linear_scaled = SVC(kernel='linear', C=4.0, random_state=42)
svm_classifier_linear_scaled.fit(Xtrain_scaled, y_train)
y_pred_SVM_linear_scaled = svm_classifier_linear_scaled.predict(Xtest_scaled)
print(f"Accuracy of the SVM Classifier (Kernel=Linear) : {accuracy_score(y_test, y_pred_SVM_linear_scaled):.2f}")

### Logistic Regression ###
LR_model_scaled = LogisticRegression(solver='saga',max_iter=5000)
LR_model_scaled.fit(Xtrain_scaled, y_train)
ypred_lr_scaled = LR_model_scaled.predict(Xtest_scaled)
accuracy_lr_scaled = accuracy_score(y_test, ypred_lr_scaled)
print(f"accuracy of the Logistic Regression Model: {round(accuracy_lr_scaled,2)}")

Accuracy of Random Forest Target Model: 0.98
Accuracy of the XGboost Model: 97.60%
Accuracy of the SVM Classifier (Kernel=Poly): 0.96
Accuracy of the SVM Classifier (Kernel=Linear) : 0.97
accuracy of the Logistic Regression Model: 0.97


In [32]:
def evasion_attack_scaled(adversarial_samples_scaled,
                   RF_model_scaled,
                   XGB_Model_scaled, 
                   SVM_poly_model_scaled, 
                   SVM_linear_model_scaled, 
                   Logistic_reg_model_scaled):
    
    predictions = {
        "Random_forest_scaled":[],
        "XGB_scaled":[],
        "SVM_poly_scaled": [], 
        "SVM_Linear_scaled":[], 
        "Logistic_Regression_scaled":[]   
    }
    
    models = {
        "Random_forest_scaled": RF_model_scaled,
        "XGB_scaled":XGB_Model_scaled,
        "SVM_poly_scaled": SVM_poly_model_scaled, 
        "SVM_Linear_scaled":SVM_linear_model_scaled, 
        "Logistic_Regression_scaled":Logistic_reg_model_scaled
    }
    
    for name, model in models.items():
        preds = model.predict(adversarial_samples_scaled)
        predictions[name] = preds.tolist()
            
    
    return predictions

In [33]:
asamples_scaled = scaler.transform(asamples)

In [34]:

res_scaled = evasion_attack_scaled(asamples_scaled, classifier_rf_scaled, model_scaled, svm_classifier_scaled, svm_classifier_linear_scaled, LR_model_scaled)
evaded_ids=[]
evaded_samples = []
original_positions = []
for model_name, preds in res_scaled.items():
    print(f"\n {model_name}: predictions")
    total = len(preds)
    count_1 = sum(1 for p in preds if p==1)
    count_2 = total-count_1 
    
    model_evaded_positions = []
    model_evaded_samples = []
    
    for i, p in enumerate(preds): 
        if p == 0: 
           xtrain_position = adv_idx[i] 
           original_position = idx_train[xtrain_position]
           model_evaded_positions.append(original_position)
            
    if model_evaded_positions:
        for og_position in model_evaded_positions:
            og_sample = data.iloc[og_position]['Sample Name']
            model_evaded_samples.append(og_sample)
        evaded_samples.extend(model_evaded_samples)
        original_positions.extend(model_evaded_positions)
    else: 
        print(f"\n No samples evaded for {model_name}")
    
    print(f"\n {model_name} evasion rate: {count_2}/{total} samples evaded")
    
    sample_names = [s.rsplit('_', 1)[0] for s in model_evaded_samples]
    print("\n evaded samples: ", Counter(sample_names)) 


 Random_forest_scaled: predictions

 No samples evaded for Random_forest_scaled

 Random_forest_scaled evasion rate: 0/370 samples evaded

 evaded samples:  Counter()

 XGB_scaled: predictions

 No samples evaded for XGB_scaled

 XGB_scaled evasion rate: 0/370 samples evaded

 evaded samples:  Counter()

 SVM_poly_scaled: predictions

 No samples evaded for SVM_poly_scaled

 SVM_poly_scaled evasion rate: 0/370 samples evaded

 evaded samples:  Counter()

 SVM_Linear_scaled: predictions

 No samples evaded for SVM_Linear_scaled

 SVM_Linear_scaled evasion rate: 0/370 samples evaded

 evaded samples:  Counter()

 Logistic_Regression_scaled: predictions

 No samples evaded for Logistic_Regression_scaled

 Logistic_Regression_scaled evasion rate: 0/370 samples evaded

 evaded samples:  Counter()
